In [1]:
from pc_skeletor import LBC

import networkx as nx
import matplotlib.pyplot as plt


from Utils.Utils import load_point_cloud
from Utils.plot_tools import  ResultsPlotter
import numpy as np
import open3d as o3d


Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


In [2]:
lbc = LBC(point_cloud=r"G:\Projects\TreeCanopyLidar\PyTLidar\test_output\CC#20.pcd",
            init_contraction = 0.2,
            init_attraction = 0.5,
            max_contraction = 2048,
            max_attraction = 1024,
            step_wise_contraction_amplification= 'auto',
            termination_ratio = 0.003,
            down_sample = -1,
            max_iteration_steps = 20,
            filter_nb_neighbors = 20,
            filter_std_ratio = 2.0,
            debug = False,
            verbose = False)
lbc.extract_skeleton()

2026-01-17 18:06:24,904 - PCD #points: 513
Volume ratio: 0.09640526682337593. Contraction weights: 3.5198053594443834. Attraction weights: 980.5237691537258. Progress LBC:  95%|█████████▌| 19/20 [00:00<00:00, 78.25it/s]2026-01-17 18:06:25,183 - Contraction is Done.
Volume ratio: 0.09640526682337593. Contraction weights: 3.5198053594443834. Attraction weights: 980.5237691537258. Progress LBC: 100%|██████████| 20/20 [00:00<00:00, 74.37it/s]

Contraction is Done.


array([[-1.11102634, -0.04012254, -1.32969611],
       [-1.15904168, -0.11789888, -1.28799606],
       [-1.14923511, -0.10751515, -1.2935852 ],
       ...,
       [-1.6793473 , -0.32676111, -1.22393115],
       [-1.6589096 , -0.35561243, -1.25936357],
       [-1.65617034, -0.35899391, -1.26323427]], shape=(513, 3))

In [3]:
o3d.visualization.draw_geometries([lbc.contracted_point_cloud])

In [6]:
lbc.extract_topology()

RUNNING NEW 


LineSet with 38 lines.

In [5]:
test = lbc.contracted_point_cloud.voxel_down_sample(voxel_size=0.1)

# lbc.contracted_point_cloud.random_down_sample
# lbc.contracted_point_cloud.uniform_down_sample

In [6]:
o3d.visualization.draw_geometries([test])

In [8]:
def plot_graph(G):
    G.add_edges_from([
        (0, 1),
        (1, 2),
        (2, 3),
        (3, 0),
        (0, 2),
    ])

    # 2. Get 3D positions for each node
    # If you already have 3D coords, skip this and use your own dict {node: (x,y,z)}
    pos = nx.spring_layout(G, dim=3, seed=0)  # 3D layout

    # 3. Map nodes to indices and build point array
    nodes = list(G.nodes())
    node_index = {n: i for i, n in enumerate(nodes)}

    points = np.array([pos[n] for n in nodes], dtype=float)  # shape (N, 3)

    # 4. Build line index array from edges
    lines = np.array([
        [node_index[u], node_index[v]]
        for u, v in G.edges()
    ], dtype=int)  # shape (M, 2)

    # 5. Create Open3D LineSet
    line_set = o3d.geometry.LineSet()
    line_set.points = o3d.utility.Vector3dVector(points)
    line_set.lines  = o3d.utility.Vector2iVector(lines)

    # Optional: color edges
    colors = np.tile(np.array([[0.2, 0.6, 1.0]]), (lines.shape[0], 1))
    line_set.colors = o3d.utility.Vector3dVector(colors)

    # 6. Visualize
    o3d.visualization.draw_geometries([line_set])

    return (line_set.points, line_set.lines)

In [9]:
lbc.skeleton = lbc.contracted_point_cloud.voxel_down_sample(voxel_size=0.1)
# lbc.skeleton = lbc.contracted_point_cloud.voxel_down_sample(voxel_size=0.0001)
# lbc.skeleton = lbc.contracted_point_cloud.random_down_sample(sampling_ratio=0.1)
skeleton_points = np.asarray(lbc.skeleton.points)

# o3d.visualization.draw_geometries([lbc.skeleton])



lbc.skeleton_graph = lbc._extract_skeletal_graph(skeletal_points=skeleton_points)

plot_graph(lbc.skeleton_graph)


# print(lbc.skeleton_graph.edges)

lbc.topology_graph, topology_points = lbc._simplify_graph(graph=lbc.skeleton_graph)

points, lines = plot_graph(lbc.topology_graph)

lbc.topology.points = points
lbc.topology.lines = lines


# # o3d.visualization.draw_geometries([lbc.skeleton_graph])

# lbc.topology_graph, topology_points = lbc._simplify_graph(graph=lbc.skeleton_graph)

# lbc.topology = o3d.geometry.LineSet()
# lbc.topology.points = o3d.utility.Vector3dVector(lbc.skeleton_graph.nodes)
# lbc.topology.lines = o3d.utility.Vector2iVector(list((lbc.skeleton_graph.edges())))

# o3d.visualization.draw_geometries([lbc.topology])


In [10]:
lbc.visualize()